<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 15



<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать базовый класс OrderLine в C#, который будет представлять информацию о строке заказа, содержащей детали одного товара в заказе. На основе этого класса разработать 2-3 производных класса, демонстрирующих принципы наследования и полиморфизма. В каждом из классов должны быть реализованы новые атрибуты и методы, а также переопределены некоторые методы базового класса для демонстрации полиморфизма.
1. СтандартнаяСтрока (StandardLine): Должна содержать дополнительные атрибуты, такие как Количество единиц (Units). Метод CalculateTotal() должен быть переопределен для учета количества единиц при расчете общей стоимости.
2. СпециальнаяСтрока (SpecialLine): Должна содержать дополнительные атрибуты, такие как Скидка (Discount). Метод UpdatePrice() должен быть переопределен для применения скидки к цене товара.
3. БесплатнаяСтрока (FreeLine) (если требуется третий класс): Должна содержать дополнительные атрибуты, такие как Предварительный платеж (Prepayment). Метод CalculateTotal() должен быть переопределен для учета предварительного плата при расчете общей стоимости.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) исользуйтие в проекте коллекции, делегаты, события.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [1]:
using System;
using System.Collections.Generic;
using System.Linq;

public delegate void PriceChangedHandler(OrderLine line, decimal oldPrice, decimal newPrice);
public delegate void OrderHandler(string message);
public delegate bool LineFilter(OrderLine line);

public class OrderLine
{
    private static int nextId = 1;

    public int Id { get; }
    public string ProductName { get; set; }
    public decimal Price { get; protected set; }
    public string Category { get; set; }
    public bool IsConfirmed { get; private set; }

    public event PriceChangedHandler PriceChanged;

    public OrderLine(string productName, decimal price, string category)
    {
        Id = nextId++;
        ProductName = productName;
        Price = price;
        Category = category;
        IsConfirmed = false;
    }

    public virtual decimal CalculateTotal()
    {
        return Price;
    }

    public virtual void UpdatePrice(decimal newPrice)
    {
        decimal oldPrice = Price;
        Price = newPrice;
        PriceChanged?.Invoke(this, oldPrice, Price);
    }

    public void Confirm()
    {
        IsConfirmed = true;
    }

    public string GetShortDescription()
    {
        return $"#{Id} {ProductName} ({Category})";
    }

    public virtual void DisplayInfo()
    {
        Console.WriteLine($"{GetShortDescription()}, Цена: {Price:F2}, Итого: {CalculateTotal():F2}, Подтверждена: {IsConfirmed}");
    }
}

public class StandardLine : OrderLine
{
    public int Units { get; set; }
    public decimal Weight { get; set; }

    public StandardLine(string productName, decimal price, string category, int units, decimal weight)
        : base(productName, price, category)
    {
        Units = units;
        Weight = weight;
    }

    public override decimal CalculateTotal()
    {
        return Price * Units;
    }

    public decimal CalculateShipping()
    {
        return Weight * Units * 50m;
    }

    public override void DisplayInfo()
    {
        base.DisplayInfo();
        Console.WriteLine($"Количество единиц: {Units}, Вес единицы: {Weight} кг, Доставка: {CalculateShipping():F2}");
    }
}

public class SpecialLine : OrderLine
{
    public decimal Discount { get; set; }
    public string PromoCode { get; set; }
    public decimal OriginalPrice { get; private set; }

    public SpecialLine(string productName, decimal price, string category, decimal discount, string promoCode)
        : base(productName, price, category)
    {
        Discount = discount;
        PromoCode = promoCode;
        OriginalPrice = price;
        Price = price * (100 - Discount) / 100;
    }

    public override void UpdatePrice(decimal newPrice)
    {
        OriginalPrice = newPrice;
        base.UpdatePrice(newPrice * (100 - Discount) / 100);
    }

    public decimal GetSavings()
    {
        return OriginalPrice - Price;
    }

    public override void DisplayInfo()
    {
        base.DisplayInfo();
        Console.WriteLine($"Скидка: {Discount}%, Промокод: {PromoCode}, Экономия: {GetSavings():F2}");
    }
}

public class FreeLine : OrderLine
{
    public decimal Prepayment { get; set; }
    public string Reason { get; set; }

    public FreeLine(string productName, decimal price, string category, decimal prepayment, string reason)
        : base(productName, price, category)
    {
        Prepayment = prepayment;
        Reason = reason;
    }

    public override decimal CalculateTotal()
    {
        return Math.Max(0m, Price - Prepayment);
    }

    public bool IsFullyPaid()
    {
        return Prepayment >= Price;
    }

    public override void DisplayInfo()
    {
        base.DisplayInfo();
        Console.WriteLine($"Предварительный платёж: {Prepayment:F2}, Причина: {Reason}, Оплачена полностью: {IsFullyPaid()}");
    }
}

public class Order
{
    private List<OrderLine> lines = new List<OrderLine>();
    private Dictionary<int, OrderLine> linesById = new Dictionary<int, OrderLine>();
    private HashSet<string> categories = new HashSet<string>();

    public string Customer { get; set; }

    public event OrderHandler LineAdded;
    public event OrderHandler OrderChanged;
    public event OrderHandler OrderConfirmed;

    public Order(string customer)
    {
        Customer = customer;
    }

    public void AddLine(OrderLine line)
    {
        lines.Add(line);
        linesById.Add(line.Id, line);
        categories.Add(line.Category);
        line.PriceChanged += OnLinePriceChanged;
        LineAdded?.Invoke($"В заказ добавлена строка {line.GetShortDescription()}");
    }

    public bool RemoveLine(int id)
    {
        if (linesById.TryGetValue(id, out OrderLine line))
        {
            line.PriceChanged -= OnLinePriceChanged;
            lines.Remove(line);
            linesById.Remove(id);
            OrderChanged?.Invoke($"Из заказа удалена строка {line.GetShortDescription()}");
            return true;
        }
        return false;
    }

    public OrderLine FindById(int id)
    {
        if (linesById.TryGetValue(id, out OrderLine line))
        {
            return line;
        }
        return null;
    }

    public List<OrderLine> Filter(LineFilter filter)
    {
        return lines.Where(line => filter(line)).ToList();
    }

    public decimal GetTotal()
    {
        return lines.Sum(line => line.CalculateTotal());
    }

    public void ConfirmAll()
    {
        lines.ForEach(line => line.Confirm());
        OrderConfirmed?.Invoke($"Заказ клиента {Customer} подтверждён, строк: {lines.Count}");
    }

    public void DisplayCategories()
    {
        Console.WriteLine("Категории в заказе: " + string.Join(", ", categories));
    }

    public void DisplayInfo()
    {
        Console.WriteLine($"Заказ клиента: {Customer}");
        foreach (var line in lines)
        {
            line.DisplayInfo();
            Console.WriteLine();
        }
        Console.WriteLine($"Общая сумма заказа: {GetTotal():F2}");
    }

    private void OnLinePriceChanged(OrderLine line, decimal oldPrice, decimal newPrice)
    {
        OrderChanged?.Invoke($"Цена {line.ProductName} изменена: {oldPrice:F2} -> {newPrice:F2}");
    }
}

Order order = new Order("Иван Петров");

order.LineAdded += message => Console.WriteLine("[Добавление] " + message);
order.OrderChanged += message => Console.WriteLine("[Изменение] " + message);
order.OrderConfirmed += message => Console.WriteLine("[Подтверждение] " + message);

var laptop = new StandardLine("Ноутбук", 50000m, "Электроника", 2, 2.5m);
var mouse = new SpecialLine("Мышь", 1500m, "Аксессуары", 10m, "SALE10");
var gift = new FreeLine("Подарочный набор", 2000m, "Подарки", 500m, "Акция");
var cable = new StandardLine("Кабель", 300m, "Аксессуары", 3, 0.1m);

order.AddLine(laptop);
order.AddLine(mouse);
order.AddLine(gift);
order.AddLine(cable);

Console.WriteLine("\n=== Информация о заказе ===");
order.DisplayInfo();

Console.WriteLine("\n=== Категории (HashSet) ===");
order.DisplayCategories();

Console.WriteLine("\n=== Поиск по Id (Dictionary) ===");
OrderLine found = order.FindById(2);
if (found != null)
{
    found.DisplayInfo();
}
else
{
    Console.WriteLine("Строка не найдена");
}

Console.WriteLine("\n=== Фильтрация через делегат и лямбду ===");
LineFilter expensive = line => line.CalculateTotal() > 1000m;
foreach (var line in order.Filter(expensive))
{
    Console.WriteLine(line.GetShortDescription());
}

Console.WriteLine("\n=== Полиморфизм и проверка типа ===");
foreach (var line in order.Filter(l => true))
{
    if (line is StandardLine standard)
    {
        Console.WriteLine($"{standard.ProductName}: доставка {standard.CalculateShipping():F2}");
    }
    else if (line is SpecialLine special)
    {
        Console.WriteLine($"{special.ProductName}: экономия {special.GetSavings():F2}");
    }
    else if (line is FreeLine free)
    {
        Console.WriteLine($"{free.ProductName}: оплачена полностью - {free.IsFullyPaid()}");
    }
}

Console.WriteLine("\n=== Изменение цены (событие) ===");
mouse.UpdatePrice(2000m);
laptop.UpdatePrice(48000m);

Console.WriteLine("\n=== Удаление строки ===");
order.RemoveLine(cable.Id);

Console.WriteLine("\n=== Подтверждение заказа ===");
order.ConfirmAll();
Console.WriteLine();
order.DisplayInfo();

The below script needs to be able to find the current output cell; this is an easy method to get it.

[Добавление] В заказ добавлена строка #1 Ноутбук (Электроника)
[Добавление] В заказ добавлена строка #2 Мышь (Аксессуары)
[Добавление] В заказ добавлена строка #3 Подарочный набор (Подарки)
[Добавление] В заказ добавлена строка #4 Кабель (Аксессуары)

=== Информация о заказе ===
Заказ клиента: Иван Петров
#1 Ноутбук (Электроника), Цена: 50000,00, Итого: 100000,00, Подтверждена: False
Количество единиц: 2, Вес единицы: 2,5 кг, Доставка: 250,00

#2 Мышь (Аксессуары), Цена: 1350,00, Итого: 1350,00, Подтверждена: False
Скидка: 10%, Промокод: SALE10, Экономия: 150,00

#3 Подарочный набор (Подарки), Цена: 2000,00, Итого: 1500,00, Подтверждена: False
Предварительный платёж: 500,00, Причина: Акция, Оплачена полностью: False

#4 Кабель (Аксессуары), Цена: 300,00, Итого: 900,00, Подтверждена: False
Количество единиц: 3, Вес единицы: 0,1 кг, Доставка: 15,00

Общая сумма заказа: 103750,00

=== Категории (HashSet) ===
Категории в заказе: Электроника, Аксессуары, Подарки

=== Поиск по Id (Dictionary